# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [12]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

client ready: gpt-4o-mini


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [13]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "사내 챗봇을 한 문장으로 소개해줘."},
    ],
    temperature=0.7,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

사내 챗봇은 직원들의 질문에 신속하게 답변하고 업무 효율성을 높이는 인공지능 기반의 대화형 도구입니다.
usage: CompletionUsage(completion_tokens=35, prompt_tokens=39, total_tokens=74, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [14]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM 을 3줄로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

대형 언어 모델(LLM)은 방대한 양의 데이터를 기반으로 훈련되어 자연어를 이해하고 생성하는 인공지능 시스템입니다. 다양한 언어 처리 작업을 수행할 수 있으며, 질문 답변, 텍스트 생성, 번역 등 다양한 용도로 활용됩니다. 이러한 모델은 패턴 인식을 통해 인간과 유사한 방식으로 언어를 생성하지만, 이해의 깊이는 제한적일 수 있습니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [15]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    52
0    48
Name: count, dtype: int64


,clean_text,label
0,사진과 색이 완전히 달라서 실망했어요,0
1,포장도 꼼꼼하고 품질이 기대 이상이에요,1
2,광고 지금 구매하면 사은품 증정 포장도 꼼꼼하고 품질이 기대 이상이에요,1
3,사이즈도 딱 맞고 마감이 깔끔해요,1
4,배송이 일주일이나 걸렸습니다 별로,0


In [16]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

라벨: 0 / 토큰: 82


In [17]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8438


### sklearn(실습 2) 과 비교 — 정확도

In [20]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from time import perf_counter
import numpy as np
import pandas as pd

# 1) sklearn 모델 학습 및 평가 (샘플과 동일한 테스트 셋 사용)
train_df = df[~df['clean_text'].isin(sample['clean_text'])].copy()
if len(train_df) < 10:
    train_df = df.sample(frac=0.8, random_state=0)
X_train, y_train = train_df['clean_text'], train_df['label']
X_test, y_test = sample['clean_text'], sample['label']

vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_t = vec.fit_transform(X_train)
X_test_t = vec.transform(X_test)
clf = LogisticRegression(max_iter=1000)
t0 = perf_counter()
clf.fit(X_train_t, y_train)
t1 = perf_counter()
train_time = t1 - t0
t0 = perf_counter()
preds_skl = clf.predict(X_test_t)
t1 = perf_counter()
predict_time_skl = t1 - t0
acc_skl = accuracy_score(y_test, preds_skl)
print(f"sklearn 정확도: {acc_skl:.3f}")
print(classification_report(y_test, preds_skl, digits=3))

# 2) LLM 재실행: 시간·토큰 측정 (주의: API 호출이 재발생합니다)
preds_llm = []
tokens_llm = 0
t0 = perf_counter()
for t in sample["clean_text"]:
    label, used = classify(t)
    preds_llm.append(label)
    tokens_llm += used
t1 = perf_counter()
total_time_llm = t1 - t0
acc_llm = accuracy_score(sample["label"], preds_llm)
print(f"LLM 정확도: {acc_llm:.3f}")
print(classification_report(sample["label"], preds_llm, digits=3))
print(f"LLM 총 토큰: {tokens_llm}")

# 3) 비교표 작성: 정확도 / 비용(100건·1만건 추산) / 속도(ms/건) / 일관성
PRICE_IN = 0.15 / 1_000_000
cost_100_llm = tokens_llm * PRICE_IN
cost_100_skl = 0.0
cost_10000_llm = cost_100_llm * 100
cost_10000_skl = 0.0

avg_ms_skl = (predict_time_skl / len(X_test)) * 1000
avg_ms_llm = (total_time_llm / len(X_test)) * 1000

comparison = pd.DataFrame({
    'method': ['sklearn', 'LLM'],
    'accuracy': [acc_skl, acc_llm],
    'cost_per_100_$': [cost_100_skl, cost_100_llm],
    'cost_per_10k_$': [cost_10000_skl, cost_10000_llm],
    'avg_ms_per_sample': [avg_ms_skl, avg_ms_llm],
    'consistency': ['deterministic', 'mostly-deterministic (temp=0)'],
})
display(comparison)

# 저장해두고 다음 셀에서 비용/한줄결론 출력에 사용
comparison_table = comparison
conclusion = "대량·단순 분류는 sklearn(ML)을 권장, 유연·복잡 추론은 LLM을 고려."


sklearn 정확도: 0.820
              precision    recall  f1-score   support

           0      1.000     0.625     0.769        48
           1      0.743     1.000     0.852        52

    accuracy                          0.820       100
   macro avg      0.871     0.812     0.811       100
weighted avg      0.866     0.820     0.813       100

LLM 정확도: 1.000
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        48
           1      1.000     1.000     1.000        52

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100

LLM 총 토큰: 8438


,method,accuracy,cost_per_100_$,cost_per_10k_$,avg_ms_per_sample,consistency
0,sklearn,0.82,0.000000,0.00000,0.003544,deterministic
1,LLM,1.00,0.001266,0.12657,703.019805,mostly-deterministic (temp=0)


## 3) Task 3 — 비용 측정 + 1만 건 추산

In [ ]:
# gpt-4o-mini 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.15/M, 출력 0.60/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens_llm / len(sample)
cost_100   = tokens_llm * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
print("한 줄 결론:", conclusion)
# TODO: ML vs LLM — '언제 무엇을 쓸지' 한 줄 결론을 적는다


평균 토큰/건: 84.4
100건 비용: $0.0013
1만 건 추산: $0.13


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*

In [22]:
import json
import re

반어_리뷰 = [
    "와 정말 대단해요, 배송도 2주나 걸리고 고장까지 났어요~",
    "품질 최고예요~ 환불하고 싶을 만큼",
    "빠른 배송 감사합니다 일주일이나 걸려서요",
]

SYSTEM_CONF = (
    "당신은 한국어 리뷰 감성 분류기입니다. 입력 리뷰가 긍정이면 1, 부정이면 0으로 판단합니다.\n"
    "출력은 반드시 JSON 형식으로만 하세요. 예: {\"label\": 1, \"confidence\": 0.87, \"reason\": \"한 문장 근거\"}\n"
    "다음 예시를 따라 응답하세요:\n"
    "예1 입력: '배송이 빨라서 좋아요' -> {\"label\": 1, \"confidence\": 0.98, \"reason\": \"직접적 긍정 표현\"}\n"
    "예2 입력: '품질 최고예요~ 환불하고 싶을 만큼' -> {\"label\": 0, \"confidence\": 0.95, \"reason\": \"긍정어+환불표현의 모순(반어)\"}\n"
    "예3 입력: '와 정말 대단해요, 배송도 2주나 걸리고 고장까지 났어요~' -> {\"label\": 0, \"confidence\": 0.93, \"reason\": \"표면적 칭찬 + 배송지연/고장 언급의 모순(반어)\"}"
)


def classify_with_confidence(text: str) -> tuple[int, float, str, int]:
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_CONF},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    label = int(data.get("label", 0))
    confidence = float(data.get("confidence", 0.0))
    reason = data.get("reason", "")
    # usage may be dict-like or object-like
    try:
        tokens_used = resp.usage.total_tokens
    except Exception:
        try:
            tokens_used = resp.usage.get('total_tokens', 0)
        except Exception:
            tokens_used = 0
    return label, confidence, reason, tokens_used


def postprocess_label(label: int, confidence: float, reason: str, text: str) -> tuple[int, float, str]:
    # 간단 규칙 기반 반어 감지
    sarc_indicators = [
        '환불', '고장', '일주일', '2주', '몇주', '못', '싫', '환불하고', '환불', '모순', '반어', '역설', '~', '싶을 만큼'
    ]
    reason_lower = (reason or '').lower()
    text_lower = (text or '').lower()
    if any(k in reason_lower for k in ['반어', '모순', '역설', '반어적']) or any(k in text_lower for k in sarc_indicators):
        # 모델이 긍정(1)으로 답했지만 반어 증거가 있으면 부정으로 교정
        if label == 1:
            new_conf = max(confidence, 0.85)
            return 0, new_conf, 'flipped_by_postprocess_sarcasm'
        # 모델이 부정인데도 reason에서 반어관련 근거가 있으면 confidence 보정
        return label, max(confidence, 0.75), 'confidence_boosted_by_postprocess'
    return label, confidence, 'no_change'

# 실행 및 출력
results = []
for r in 반어_리뷰:
    try:
        lbl, conf, reason, used = classify_with_confidence(r)
    except Exception as e:
        print("분류 중 오류:", e)
        lbl, conf, reason, used = None, 0.0, '', 0
    adj_lbl, adj_conf, post_tag = postprocess_label(lbl if lbl is not None else -1, conf, reason, r)
    results.append({
        "text": r,
        "orig_label": lbl,
        "orig_confidence": conf,
        "reason": reason,
        "tokens": used,
        "adj_label": adj_lbl,
        "adj_confidence": adj_conf,
        "post_tag": post_tag,
    })

for r in results:
    orig = '긍정' if r['orig_label'] == 1 else ('부정' if r['orig_label'] == 0 else '알수없음')
    adj = '긍정' if r['adj_label'] == 1 else ('부정' if r['adj_label'] == 0 else '알수없음')
    print('리뷰:', r['text'])
    print(f"-> 모델 판정: {orig} (label={r['orig_label']}, conf={r['orig_confidence']:.2f})")
    print(f"   reason: {r['reason']}")
    print(f"   후처리: {adj} (label={r['adj_label']}, conf={r['adj_confidence']:.2f})  tag={r['post_tag']}")
    print(f"   tokens: {r['tokens']}")
    print()


리뷰: 와 정말 대단해요, 배송도 2주나 걸리고 고장까지 났어요~
-> 모델 판정: 부정 (label=0, conf=0.93)
   reason: 표면적 칭찬 + 배송지연/고장 언급의 모순(반어)
   후처리: 부정 (label=0, conf=0.93)  tag=confidence_boosted_by_postprocess
   tokens: 318

리뷰: 품질 최고예요~ 환불하고 싶을 만큼
-> 모델 판정: 부정 (label=0, conf=0.95)
   reason: 긍정어+환불표현의 모순(반어)
   후처리: 부정 (label=0, conf=0.95)  tag=confidence_boosted_by_postprocess
   tokens: 302

리뷰: 빠른 배송 감사합니다 일주일이나 걸려서요
-> 모델 판정: 부정 (label=0, conf=0.92)
   reason: 감사의 표현과 배송 지연의 모순(반어)
   후처리: 부정 (label=0, conf=0.92)  tag=confidence_boosted_by_postprocess
   tokens: 300

